# TransJAX Demo

**TransJAX** translates legacy Fortran scientific code to [JAX](https://github.com/google/jax) using multi-agent LLM pipelines powered by Claude.

This notebook walks through the full workflow:

1. [Setup](#1-setup)
2. [Static Analysis — parse & explore a Fortran codebase](#2-static-analysis)
3. [Dependency graph & translation order](#3-dependency-graph--translation-order)
4. [Translation unit decomposition](#4-translation-unit-decomposition)
5. [Agent-based translation (TranslatorAgent)](#5-agent-based-translation)
6. [Test generation (TestAgent)](#6-test-generation)
7. [Repair loop (RepairAgent)](#7-repair-loop)
8. [Full orchestrated pipeline (OrchestratorAgent)](#8-full-orchestrated-pipeline)

> **Note** — Sections 5-8 call the Anthropic API and require `ANTHROPIC_API_KEY` to be set.

## 1  Setup

In [ ]:
# Install TransJAX (skip if already installed)
# !pip install -e '.[dev]'   # from the repo root
# !pip install transjax      # from PyPI once published

import os, json, textwrap, tempfile
from pathlib import Path

# Optionally set your key here if not already in the environment
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."

has_api_key = bool(os.environ.get("ANTHROPIC_API_KEY"))
print("API key present:", has_api_key)
print("Sections 5-8 require an API key; sections 2-4 run fully offline.")

### Sample Fortran project

We create a tiny self-contained Fortran project in a temp directory so the demo runs without any external files.

In [ ]:
FORTRAN_CONSTANTS = textwrap.dedent("""\
    module physics_constants
      implicit none

      ! Fundamental physical constants
      real, parameter :: PI        = 3.14159265358979
      real, parameter :: GRAVITY   = 9.80665          ! m/s^2
      real, parameter :: BOLTZMANN = 1.380649e-23     ! J/K
      real, parameter :: AVOGADRO  = 6.02214076e23    ! mol^-1

    end module physics_constants
""")

FORTRAN_MATH = textwrap.dedent("""\
    module simple_math
      use physics_constants, only: PI
      implicit none

      contains

      ! Add two real numbers
      subroutine add(a, b, result)
        real, intent(in)  :: a, b
        real, intent(out) :: result
        result = a + b
      end subroutine add

      ! Compute the square of x
      function square(x) result(y)
        real, intent(in) :: x
        real :: y
        y = x * x
      end function square

      ! Convert degrees to radians
      function deg_to_rad(deg) result(rad)
        real, intent(in) :: deg
        real :: rad
        rad = deg * PI / 180.0
      end function deg_to_rad

    end module simple_math
""")

FORTRAN_THERMO = textwrap.dedent("""\
    module thermodynamics
      use physics_constants, only: BOLTZMANN
      use simple_math,       only: square
      implicit none

      contains

      ! Ideal gas law: P = nRT/V
      subroutine ideal_gas_pressure(n, R, T, V, P)
        real, intent(in)  :: n, R, T, V
        real, intent(out) :: P
        P = n * R * T / V
      end subroutine ideal_gas_pressure

      ! Root-mean-square speed of gas molecules: v_rms = sqrt(3kT/m)
      function rms_speed(T, m) result(v)
        real, intent(in) :: T, m   ! temperature (K), molecular mass (kg)
        real :: v
        v = sqrt(3.0 * BOLTZMANN * T / m)
      end function rms_speed

      ! Kinetic energy: E = 0.5 * m * v^2
      function kinetic_energy(m, v) result(E)
        real, intent(in) :: m, v
        real :: E
        E = 0.5 * m * square(v)
      end function kinetic_energy

    end module thermodynamics
""")

# Write files to a temporary directory
tmp_dir = tempfile.mkdtemp(prefix="transjax_demo_")
src_dir = Path(tmp_dir) / "src"
src_dir.mkdir()

(src_dir / "physics_constants.f90").write_text(FORTRAN_CONSTANTS)
(src_dir / "simple_math.f90").write_text(FORTRAN_MATH)
(src_dir / "thermodynamics.f90").write_text(FORTRAN_THERMO)

print(f"Sample project written to: {tmp_dir}")
print("Files:", [f.name for f in src_dir.iterdir()])

---
## 2  Static Analysis

The analyzer parses Fortran source files and builds a rich representation of modules, subroutines, functions, and their dependencies — **no API key required**.

In [ ]:
from transjax.analyzer import FortranAnalyzer, create_analyzer_for_project, quick_analyze
from transjax.analyzer.config.project_config import FortranProjectConfig

# Option A — one-liner helper
results = quick_analyze(str(src_dir), template="generic", output_dir=None)

print("=== Quick analysis summary ===")
stats = results.get("statistics", {})
for k, v in stats.items():
    print(f"  {k}: {v}")

In [ ]:
# Option B — explicit config for finer control
config = FortranProjectConfig(
    project_name="demo_project",
    project_root=str(src_dir),
    source_dirs=[str(src_dir)],
    generate_graphs=False,   # set True to emit graph files
)

analyzer = FortranAnalyzer(config)
analysis = analyzer.analyze(save_results=False)

print("Modules found:")
for name, mod in analysis["modules"].items():
    subs  = mod.subroutines
    funcs = mod.functions
    deps  = [u["module"] for u in mod.uses]
    print(f"  {name:25s}  subroutines={subs}  functions={funcs}  uses={deps}")

In [ ]:
# Inspect entities inside a module
mod = analysis["modules"]["simple_math"]
print(f"Module '{mod.name}'  ({mod.line_count} lines)\n")
for e in mod.entities:
    print(f"  [{e.entity_type:12s}] {e.name}  (lines {e.line_start}-{e.line_end})")

---
## 3  Dependency Graph & Translation Order

TransJAX builds a directed dependency graph and performs a topological sort to determine the safest translation order (dependencies first).

In [ ]:
from transjax.analyzer.analysis.call_graph_builder import CallGraphBuilder

builder = CallGraphBuilder(config)
dep_graph = builder.build_module_dependency_graph(analysis["modules"])

print("Module dependency edges:")
for src, dst in dep_graph.edges():
    print(f"  {src}  -->  {dst}")

In [ ]:
translation_order = analyzer.get_translation_order()
print("Recommended translation order (dependencies first):")
for i, name in enumerate(translation_order, 1):
    print(f"  {i}. {name}")

In [ ]:
# Graph metrics
metrics = builder.calculate_metrics()
print("Graph metrics:")
print(json.dumps(metrics, indent=2, default=str))

In [ ]:
# Optional: draw the dependency graph (requires matplotlib + networkx)
try:
    import matplotlib.pyplot as plt
    import networkx as nx

    pos = nx.spring_layout(dep_graph, seed=42)
    plt.figure(figsize=(7, 4))
    nx.draw_networkx(
        dep_graph, pos,
        with_labels=True,
        node_color="#4A90D9",
        font_color="white",
        font_size=10,
        node_size=2000,
        arrows=True,
        arrowsize=20,
    )
    plt.title("Module Dependency Graph")
    plt.axis("off")
    plt.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib / networkx not installed — skipping visualisation.")

---
## 4  Translation Unit Decomposition

Large Fortran modules are split into *translation units* — self-contained chunks sized for a single LLM context window. Each unit gets a complexity score and effort estimate.

In [ ]:
from transjax.analyzer.analysis.translation_decomposer import TranslationUnitDecomposer

decomposer = TranslationUnitDecomposer(config)
units = decomposer.decompose_modules(analysis["modules"])

print(f"Total translation units: {len(units)}\n")
print(f"{'ID':<30} {'type':<18} {'module':<22} {'effort':<8} {'complexity'}")
print("-" * 90)
for u in units:
    print(f"{u.id:<30} {u.unit_type:<18} {u.module_name:<22} {u.estimated_effort:<8} {u.complexity_score:.2f}")

In [ ]:
# Units sorted by priority (most important first)
priority_units = decomposer.get_units_by_priority()
print("Units by descending priority:")
for u in priority_units:
    print(f"  priority={u.priority}  {u.id}  depends_on={u.depends_on}")

In [ ]:
stats = decomposer.get_statistics()
print("Decomposition statistics:")
print(json.dumps(stats, indent=2, default=str))

---
## 5  Agent-based Translation

The `TranslatorAgent` sends Fortran source to Claude and receives idiomatic JAX/Python code back.

> **Requires** `ANTHROPIC_API_KEY`.

In [ ]:
if not has_api_key:
    print("Skipping — ANTHROPIC_API_KEY not set.")
else:
    from transjax.agents import TranslatorAgent

    output_dir = Path(tmp_dir) / "jax_output"
    output_dir.mkdir(exist_ok=True)

    translator = TranslatorAgent()

    # Translate the simplest module first
    result = translator.translate_module(
        module_name="simple_math",
        fortran_file=src_dir / "simple_math.f90",
        output_dir=output_dir,
    )

    print(f"Module       : {result.module_name}")
    print(f"Notes preview: {result.translation_notes[:200]}...\n")
    print("--- Generated JAX code (first 60 lines) ---")
    for line in result.physics_code.splitlines()[:60]:
        print(line)

In [ ]:
if not has_api_key:
    print("Skipping.")
else:
    # Save outputs and inspect the file layout
    saved_paths = result.save_structured(output_dir)
    print("Saved files:")
    for kind, path in saved_paths.items():
        print(f"  {kind}: {path}")

    # Cost estimate for this translation call
    cost = translator.get_cost_estimate()
    print("\nAPI cost estimate:")
    print(json.dumps(cost, indent=2))

---
## 6  Test Generation

`TestAgent` inspects the translated Python code and generates a full `pytest` suite with synthetic test data, edge cases, and assertions.

> **Requires** `ANTHROPIC_API_KEY`.

In [ ]:
if not has_api_key:
    print("Skipping — ANTHROPIC_API_KEY not set.")
else:
    from transjax.agents import TestAgent

    test_agent = TestAgent()

    test_result = test_agent.generate_tests(
        module_name="simple_math",
        python_code=result.physics_code,
        output_dir=output_dir,
        num_test_cases=8,
        include_edge_cases=True,
    )

    print(f"Module: {test_result.module_name}")
    print("\n--- Generated pytest file (first 60 lines) ---")
    for line in test_result.pytest_file.splitlines()[:60]:
        print(line)

---
## 7  Repair Loop

When generated tests fail, `RepairAgent` analyses the failure, determines the root cause, and rewrites the Python code iteratively until all tests pass.

> **Requires** `ANTHROPIC_API_KEY`.

In [ ]:
if not has_api_key:
    print("Skipping — ANTHROPIC_API_KEY not set.")
else:
    from transjax.agents import RepairAgent

    # Simulate a broken translation to demonstrate the repair agent
    broken_python = textwrap.dedent("""\
        import jax.numpy as jnp

        def add(a: float, b: float) -> float:
            return a - b  # intentional bug: should be a + b

        def square(x: float) -> float:
            return x * x

        def deg_to_rad(deg: float) -> float:
            return deg * jnp.pi / 180.0
    """)

    fake_test_report = textwrap.dedent("""\
        FAILED test_simple_math.py::test_add - AssertionError:
          add(2.0, 3.0) returned -1.0, expected 5.0
        FAILED test_simple_math.py::test_add_zero - AssertionError:
          add(0.0, 5.0) returned -5.0, expected 5.0
        PASSED test_simple_math.py::test_square
        PASSED test_simple_math.py::test_deg_to_rad
        2 failed, 2 passed
    """)

    repair_agent = RepairAgent()

    repair_result = repair_agent.repair_translation(
        module_name="simple_math",
        fortran_code=FORTRAN_MATH,
        failed_python_code=broken_python,
        test_report=fake_test_report,
        output_dir=output_dir,
    )

    print(f"Repair iterations : {repair_result.iterations}")
    print(f"All tests passed  : {repair_result.all_tests_passed}")
    print(f"\nRoot cause analysis:\n{repair_result.root_cause_analysis}")
    print("\n--- Corrected code ---")
    print(repair_result.corrected_python_code)

---
## 8  Full Orchestrated Pipeline

`OrchestratorAgent` ties everything together: static analysis → dependency ordering → translation → test generation → repair.

> **Requires** `ANTHROPIC_API_KEY`.

In [ ]:
if not has_api_key:
    print("Skipping — ANTHROPIC_API_KEY not set.")
else:
    from transjax.agents import OrchestratorAgent

    orch_output = Path(tmp_dir) / "orch_output"

    orch = OrchestratorAgent(
        fortran_dir=src_dir,
        output_dir=orch_output,
        max_repair_iterations=3,
        skip_tests=False,
        skip_repair=False,
    )

    summary = orch.run()

    print("=== Translation summary ===")
    print(f"Total modules    : {summary['total_modules']}")
    print(f"Translated       : {summary['translated_count']}")
    print(f"Tests passed     : {summary.get('tests_passed_count', 'n/a')}")
    print()
    print("Per-module status:")
    for name, status in summary.get("module_statuses", {}).items():
        print(f"  {name:25s}  {status}")

In [ ]:
if not has_api_key:
    print("Skipping.")
else:
    print("Output directory layout:")
    for p in sorted(orch_output.rglob("*")):
        rel = p.relative_to(orch_output)
        indent = "  " * (len(rel.parts) - 1)
        print(f"{indent}{p.name}{'/' if p.is_dir() else ''}")

---
## CLI Quick-Reference

TransJAX ships a CLI for the same operations:

```bash
# Analyse only (no API key needed)
transjax analyze /path/to/fortran

# Full translate + test + repair
export ANTHROPIC_API_KEY="sk-ant-..."
transjax convert /path/to/fortran -o ./jax_output

# Useful flags
transjax convert /path/to/fortran \
    -o ./jax_output \
    --model claude-opus-4-6 \
    --max-repair-iterations 5 \
    --modules simple_math,thermodynamics \
    --verbose

# Inspect resolved configuration
transjax show-config
```

---
## Cleanup

In [ ]:
import shutil
shutil.rmtree(tmp_dir, ignore_errors=True)
print("Temporary files removed.")